# 02 — Data Cleaning and Feature Engineering
**Project:** Student Attendance and Internal Assessment Performance Analysis  
**Input:** `data/original/student_performance.csv`  
**Output:** `data/cleaned/student_performance_cleaned.csv`  
**Rule:** The original dataset is never modified.

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import os

In [2]:
# Cell 2 — File paths
BASE = os.path.join('..', '')  # project root relative to notebooks/
ORIG_PATH  = os.path.join('..', 'data', 'original', 'student_performance.csv')
CLEAN_DIR  = os.path.join('..', 'data', 'cleaned')
CLEAN_PATH = os.path.join(CLEAN_DIR, 'student_performance_cleaned.csv')

os.makedirs(CLEAN_DIR, exist_ok=True)

In [3]:
# Cell 3 — Load original dataset (read-only)
df = pd.read_csv(ORIG_PATH)
print('Original shape:', df.shape)
df.head()

Original shape: (300, 16)


,student_id,name,age,gender,quiz1_marks,quiz2_marks,quiz3_marks,total_assignments,assignments_submitted,midterm_marks,final_marks,previous_gpa,total_lectures,lectures_attended,total_lab_sessions,labs_attended
0,1,Kristina Vaughan,19,Male,8.0,5.7,7.4,5,NaN,30.0,36.5,2.57,12,4,6,1
1,2,Rodney Daniels,21,Male,10.0,7.9,4.1,5,NaN,25.4,33.0,2.40,12,1,6,5
2,3,Jose Nash,19,Female,7.5,1.2,0.3,5,NaN,14.4,24.8,2.99,12,0,6,0
3,4,Nicole Martin,21,Male,5.2,2.5,9.9,5,NaN,17.7,41.0,1.68,12,9,6,0
4,5,Shelby Smith,21,Female,5.9,6.3,2.0,5,NaN,23.8,31.0,2.53,12,7,6,4


In [4]:
# Cell 4 — Drop assignments_submitted (100% missing)
# This column has no usable values across all 300 rows.
print('Missing in assignments_submitted:', df['assignments_submitted'].isnull().sum(), 'out of', len(df))

df = df.drop(columns=['assignments_submitted'])
print('Shape after drop:', df.shape)

Missing in assignments_submitted: 300 out of 300
Shape after drop: (300, 15)


In [5]:
# Cell 5 — Check remaining missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print('\nTotal missing cells:', missing.sum())

Missing values per column:
student_id            0
name                  0
age                   0
gender                0
quiz1_marks           0
quiz2_marks           0
quiz3_marks           0
total_assignments     0
midterm_marks         0
final_marks           0
previous_gpa          0
total_lectures        0
lectures_attended     0
total_lab_sessions    0
labs_attended         0
dtype: int64

Total missing cells: 0


In [6]:
# Cell 6 — Check duplicates
dups = df.duplicated().sum()
print('Duplicate rows:', dups)

Duplicate rows: 0


In [7]:
# Cell 7 — Validate marks ranges
print('quiz1_marks   :', df['quiz1_marks'].min(), 'to', df['quiz1_marks'].max(), '  (expected 0-10)')
print('quiz2_marks   :', df['quiz2_marks'].min(), 'to', df['quiz2_marks'].max(), '  (expected 0-10)')
print('quiz3_marks   :', df['quiz3_marks'].min(), 'to', df['quiz3_marks'].max(), '  (expected 0-10)')
print('midterm_marks :', df['midterm_marks'].min(), 'to', df['midterm_marks'].max(), ' (expected 0-30)')
print('final_marks   :', df['final_marks'].min(), 'to', df['final_marks'].max(), ' (expected 0-50)')
print('previous_gpa  :', df['previous_gpa'].min(), 'to', df['previous_gpa'].max(), '  (expected 0-4)')

quiz1_marks   : 2.8 to 10.0   (expected 0-10)
quiz2_marks   : 0.0 to 10.0   (expected 0-10)
quiz3_marks   : 0.0 to 10.0   (expected 0-10)
midterm_marks : 0.0 to 30.0  (expected 0-30)
final_marks   : 0.0 to 50.0  (expected 0-50)
previous_gpa  : 0.61 to 4.0   (expected 0-4)


In [8]:
# Cell 8 — Validate attendance counts
print('total_lectures (unique values)   :', df['total_lectures'].unique())
print('total_lab_sessions (unique values):', df['total_lab_sessions'].unique())

print('lectures_attended range:', df['lectures_attended'].min(), 'to', df['lectures_attended'].max())
print('labs_attended range    :', df['labs_attended'].min(), 'to', df['labs_attended'].max())

# No student should have attended more sessions than the total scheduled
invalid_lec = (df['lectures_attended'] > df['total_lectures']).sum()
invalid_lab = (df['labs_attended'] > df['total_lab_sessions']).sum()
print('\nStudents with lectures_attended > total_lectures:', invalid_lec)
print('Students with labs_attended > total_lab_sessions:', invalid_lab)

total_lectures (unique values)   : [12]
total_lab_sessions (unique values): [6]
lectures_attended range: 0 to 12
labs_attended range    : 0 to 6

Students with lectures_attended > total_lectures: 0
Students with labs_attended > total_lab_sessions: 0


In [9]:
# Cell 9 — Feature engineering: Lecture attendance percentage
# Formula: (lectures_attended / total_lectures) * 100
# np.where handles the zero-denominator case safely (returns 0.0 if total is 0).

df['lecture_attendance_pct'] = np.where(
    df['total_lectures'] > 0,
    (df['lectures_attended'] / df['total_lectures']) * 100,
    0.0
)

print('lecture_attendance_pct — sample values:')
print(df[['lectures_attended', 'total_lectures', 'lecture_attendance_pct']].head())

lecture_attendance_pct — sample values:
   lectures_attended  total_lectures  lecture_attendance_pct
0                  4              12               33.333333
1                  1              12                8.333333
2                  0              12                0.000000
3                  9              12               75.000000
4                  7              12               58.333333


In [10]:
# Cell 10 — Feature engineering: Lab attendance percentage
# Formula: (labs_attended / total_lab_sessions) * 100

df['lab_attendance_pct'] = np.where(
    df['total_lab_sessions'] > 0,
    (df['labs_attended'] / df['total_lab_sessions']) * 100,
    0.0
)

print('lab_attendance_pct — sample values:')
print(df[['labs_attended', 'total_lab_sessions', 'lab_attendance_pct']].head())

lab_attendance_pct — sample values:
   labs_attended  total_lab_sessions  lab_attendance_pct
0              1                   6           16.666667
1              5                   6           83.333333
2              0                   6            0.000000
3              0                   6            0.000000
4              4                   6           66.666667


In [11]:
# Cell 11 — Feature engineering: Overall attendance percentage
# Formula: (lectures_attended + labs_attended) / (total_lectures + total_lab_sessions) * 100
# Combines both lecture and lab attendance into one measure.

total_sessions = df['total_lectures'] + df['total_lab_sessions']

df['overall_attendance_pct'] = np.where(
    total_sessions > 0,
    ((df['lectures_attended'] + df['labs_attended']) / total_sessions) * 100,
    0.0
)

print('overall_attendance_pct stats:')
print(df['overall_attendance_pct'].describe().round(2))

overall_attendance_pct stats:
count    300.00
mean      50.69
std       23.28
min        0.00
25%       33.33
50%       50.00
75%       72.22
max      100.00
Name: overall_attendance_pct, dtype: float64


In [12]:
# Cell 12 — Feature engineering: Average quiz marks
# Formula: (quiz1 + quiz2 + quiz3) / 3

df['avg_quiz_marks'] = (df['quiz1_marks'] + df['quiz2_marks'] + df['quiz3_marks']) / 3

print('avg_quiz_marks stats:')
print(df['avg_quiz_marks'].describe().round(4))

avg_quiz_marks stats:


count    300.0000
mean       6.1968
std        1.2796
min        2.7000
25%        5.4333
50%        6.1500
75%        7.1083
max       10.0000
Name: avg_quiz_marks, dtype: float64


In [13]:
# Cell 13 — Preview cleaned dataset
print('Final shape:', df.shape)
print('\nColumns:', list(df.columns))
df.head()

Final shape: (300, 19)

Columns: ['student_id', 'name', 'age', 'gender', 'quiz1_marks', 'quiz2_marks', 'quiz3_marks', 'total_assignments', 'midterm_marks', 'final_marks', 'previous_gpa', 'total_lectures', 'lectures_attended', 'total_lab_sessions', 'labs_attended', 'lecture_attendance_pct', 'lab_attendance_pct', 'overall_attendance_pct', 'avg_quiz_marks']


,student_id,name,age,gender,quiz1_marks,quiz2_marks,quiz3_marks,total_assignments,midterm_marks,final_marks,previous_gpa,total_lectures,lectures_attended,total_lab_sessions,labs_attended,lecture_attendance_pct,lab_attendance_pct,overall_attendance_pct,avg_quiz_marks
0,1,Kristina Vaughan,19,Male,8.0,5.7,7.4,5,30.0,36.5,2.57,12,4,6,1,33.333333,16.666667,27.777778,7.033333
1,2,Rodney Daniels,21,Male,10.0,7.9,4.1,5,25.4,33.0,2.40,12,1,6,5,8.333333,83.333333,33.333333,7.333333
2,3,Jose Nash,19,Female,7.5,1.2,0.3,5,14.4,24.8,2.99,12,0,6,0,0.000000,0.000000,0.000000,3.000000
3,4,Nicole Martin,21,Male,5.2,2.5,9.9,5,17.7,41.0,1.68,12,9,6,0,75.000000,0.000000,50.000000,5.866667
4,5,Shelby Smith,21,Female,5.9,6.3,2.0,5,23.8,31.0,2.53,12,7,6,4,58.333333,66.666667,61.111111,4.733333


In [14]:
# Cell 14 — Save cleaned dataset
# Original file at data/original/ is NEVER modified.
df.to_csv(CLEAN_PATH, index=False)
print('Cleaned dataset saved to:', CLEAN_PATH)

# Verify by reloading
verify = pd.read_csv(CLEAN_PATH)
print('Verification — shape:', verify.shape)
print('Verification — missing values:', verify.isnull().sum().sum())

Cleaned dataset saved to:

 ..\data\cleaned\student_performance_cleaned.csv
Verification — shape: (300, 19)
Verification — missing values: 0
